In [11]:
import json, os, shutil
import numpy as np
import pandas as pd

CODE_DIR = '/kaggle/input/datasets/kaggle/meta-kaggle-code'
OUT = '/kaggle/working/corpus'
TARGET, RECENCY = 5000, 3.0   # RECENCY: newest kernels are e**3 (about 20x) likelier than oldest

print("Loading the data...")
k = pd.read_csv('/kaggle/input/datasets/kaggle/meta-kaggle/Kernels.csv',
                usecols=['Id', 'CurrentKernelVersionId', 'TotalVotes']).dropna()
k['vid'] = k['CurrentKernelVersionId'].astype('int64')

# Votes accumulate for years, so a fixed threshold favors old kernels: take the top 10% per era.
era = pd.qcut(k['vid'], 10, labels=False)
k = k[k['TotalVotes'] >= k.groupby(era)['TotalVotes'].transform(lambda v: v.quantile(0.9))]

recency = k['vid'].rank(pct=True)                      # 0 = oldest, 1 = newest
order = k.sample(n=min(len(k), TARGET * 2), weights=np.exp(RECENCY * recency), random_state=42)

print("Copying files...")
os.makedirs(OUT, exist_ok=True)
copied = 0
for vid in order['vid']:
    print(".", end="", flush=True)
    path = os.path.join(CODE_DIR, f"{vid // 1_000_000:04d}", f"{vid // 1_000 % 1_000:03d}", f"{vid}.ipynb")
    try:
        with open(path, encoding='utf-8') as f:
            meta = json.load(f).get('metadata', {})
    except (OSError, ValueError):
        continue                                        # missing or unreadable
    if (meta.get('language_info') or {}).get('name', 'python') != 'python':
        continue                                        # R and other kernels
    shutil.copy(path, f'{OUT}/{vid}.ipynb')
    copied += 1
    if copied == TARGET:
        break
        
print("")
print(f"Copied {copied} notebooks.")
shutil.make_archive('/kaggle/working/kaggle_corpus', 'zip', OUT)

Loading the data...
Copying files...
...................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................

'/kaggle/working/kaggle_corpus.zip'

In [6]:
import pandas as pd
import os

META_DIR = '/kaggle/input/datasets/kaggle/meta-kaggle'

kernels = pd.read_csv(os.path.join(META_DIR, 'Kernels.csv'), nrows=5)
versions = pd.read_csv(os.path.join(META_DIR, 'KernelVersions.csv'), nrows=5)

print("Kernels columns:", list(kernels.columns))
print("KernelVersions columns:", list(versions.columns))

Kernels columns: ['Id', 'AuthorUserId', 'CurrentKernelVersionId', 'ForkParentKernelVersionId', 'ForumTopicId', 'FirstKernelVersionId', 'CreationDate', 'EvaluationDate', 'MadePublicDate', 'IsProjectLanguageTemplate', 'CurrentUrlSlug', 'Medal', 'MedalAwardDate', 'TotalViews', 'TotalComments', 'TotalVotes']
KernelVersions columns: ['Id', 'ScriptId', 'ParentScriptVersionId', 'ScriptLanguageId', 'AuthorUserId', 'CreationDate', 'VersionNumber', 'Title', 'EvaluationDate', 'IsChange', 'TotalLines', 'LinesInsertedFromPrevious', 'LinesChangedFromPrevious', 'LinesUnchangedFromPrevious', 'LinesInsertedFromFork', 'LinesDeletedFromFork', 'LinesChangedFromFork', 'LinesUnchangedFromFork', 'TotalVotes', 'IsInternetEnabled', 'RunningTimeInMilliseconds', 'AcceleratorTypeId', 'DockerImage']


In [7]:
high_vote_kernels = kernels[kernels['TotalVotes'] > 50]
python_versions = versions[versions['ScriptLanguageId'] == 9]
targets = high_vote_kernels.merge(python_versions, left_on='Id', right_on='ScriptId', how='inner')
latest_versions = targets.groupby('ScriptId')['Id_y'].max().astype(str).tolist()[:1000]

In [8]:
print("Total high-vote kernels:", len(high_vote_kernels))
print("Total python versions:", len(python_versions))
print("Merged targets count:", len(targets))
print("Unique latest versions targeted:", len(latest_versions))

Total high-vote kernels: 1
Total python versions: 4
Merged targets count: 0
Unique latest versions targeted: 0
